# Lesson 10.7 - Production MCP Patterns
Netsetos GenAI Course v2.0 | Module 10: Tool Use & MCP

5 production pillars: rate limit, error recovery, logging, observability, security. Plus auth, multi-tenancy, versioning, health checks.


In [ ]:
print('Module 10: Tool Use & MCP')
print('Lesson 10.7: Production MCP Patterns')


## Ex 1: The Five Production Pillars

**📖 Concept:** Every production MCP server rests on five pillars: rate limiting, error recovery, structured logging, observability, and security. Miss one and it is a demo, not production.

**📝 Task:** Given the `pillars` data, print a header line then one aligned row per pillar.

**📤 Expected:** Five aligned rows, one per pillar, each naming the reason and a common tool.

In [ ]:
# ✏️ YOUR CODE HERE
pillars = [
    ('Rate limiting',     'one noisy client cannot starve others', 'token bucket per client_id'),
    ('Error recovery',     'transient failures must not cascade',  '3-state breaker per tool + retry'),
    ('Structured logging','queryable JSON, not free-text',         'JSON + W3C trace_id'),
    ('Observability',     'know which tool, which tenant, which p99', 'OTel + Prometheus + Grafana'),
    ('Security',          'auth + tool allowlist + tenant isolation', 'OAuth 2.0 + scoped tokens'),
]

# TODO: print a header, then one aligned row per pillar using:
#   f'  {pillar:18s}: {why:42s} | tool: {common_tool}'

<details><summary>💡 Hint</summary>

Iterate the tuples with `for pillar, why, common_tool in pillars:` and reuse the width specifiers `:18s` / `:42s` so the columns line up.
</details>

**✅ Criteria:** Header printed, five rows, columns aligned.

<details><summary>✅ Solution</summary>



In [ ]:
pillars = [
    ('Rate limiting',     'one noisy client cannot starve others', 'token bucket per client_id'),
    ('Error recovery',     'transient failures must not cascade',  '3-state breaker per tool + retry'),
    ('Structured logging','queryable JSON, not free-text',         'JSON + W3C trace_id'),
    ('Observability',     'know which tool, which tenant, which p99', 'OTel + Prometheus + Grafana'),
    ('Security',          'auth + tool allowlist + tenant isolation', 'OAuth 2.0 + scoped tokens'),
]
print('Five pillars every production MCP server needs:')
for pillar, why, common_tool in pillars:
    print(f'  {pillar:18s}: {why:42s} | tool: {common_tool}')

</details>

---

## Ex 2: Per-Client Token Bucket

**📖 Concept:** Rate limiting is per-CLIENT (a token bucket per client_id), not per-server. Each tier sets a bucket capacity (burst) and a refill rate (sustained throughput).

**📝 Task:** Print each tier's capacity and refill from `tiers`, then print the empty-bucket rule (429 + Retry-After).

**📤 Expected:** Three tier rows (free / pro / enterprise) plus the 429 note.

In [ ]:
# ✏️ YOUR CODE HERE
tiers = [
    ('free',       10,   1, '10 in burst, 1/sec sustained'),
    ('pro',       100,  10, '100 burst, 10/sec'),
    ('enterprise',1000,100, '1000 burst, 100/sec'),
]

# TODO: print each tier row as
#   f'  {tier:12s}: capacity={capacity:>4} refill={refill_per_sec:>3}/s | {burst}'
# then print the empty-bucket rule: 429 + Retry-After = ceil(1/refill) seconds

<details><summary>💡 Hint</summary>

Unpack four fields: `for tier, capacity, refill_per_sec, burst in tiers:`. An empty bucket returns HTTP 429 with `Retry-After = ceil(1/refill)` seconds.
</details>

**✅ Criteria:** Three rows printed, capacity/refill aligned, 429 rule stated.

<details><summary>✅ Solution</summary>



In [ ]:
tiers = [
    ('free',       10,   1, '10 in burst, 1/sec sustained'),
    ('pro',       100,  10, '100 burst, 10/sec'),
    ('enterprise',1000,100, '1000 burst, 100/sec'),
]
print('Token bucket parameters by tier (illustrative):')
for tier, capacity, refill_per_sec, burst in tiers:
    print(f'  {tier:12s}: capacity={capacity:>4} refill={refill_per_sec:>3}/s | {burst}')
print()
print('Empty bucket -> 429 + Retry-After: ceil(1/refill) seconds.')

</details>

---

## Ex 3: Per-Tool Circuit Breaker

**📖 Concept:** A circuit breaker protects each tool with three states: CLOSED (pass through), OPEN (reject instantly), and HALF_OPEN (one probe after cooldown). It trips per-TOOL, so one flaky tool cannot take down the whole server.

**📝 Task:** Print each state and its behavior from `states`, then print the per-tool note.

**📤 Expected:** Three state rows plus the per-tool reminder.

In [ ]:
# ✏️ YOUR CODE HERE
states = [
    ('CLOSED',     'normal: tool calls pass through'),
    ('OPEN',       'reject instantly with status=circuit_open; LLM decides to retry/skip'),
    ('HALF_OPEN',  'one probe call after cooldown; success -> CLOSED, fail -> OPEN'),
]

# TODO: print each state row as f'  {state:10s}: {behavior}'
# then print why the breaker trips per-TOOL not per-server

<details><summary>💡 Hint</summary>

`for state, behavior in states:` then `print(f'  {state:10s}: {behavior}')`. Tripping per-tool keeps one flaky tool from downing the whole MCP.
</details>

**✅ Criteria:** All three states printed with behavior, per-tool note included.

<details><summary>✅ Solution</summary>



In [ ]:
states = [
    ('CLOSED',     'normal: tool calls pass through'),
    ('OPEN',       'reject instantly with status=circuit_open; LLM decides to retry/skip'),
    ('HALF_OPEN',  'one probe call after cooldown; success -> CLOSED, fail -> OPEN'),
]
print('Per-tool 3-state breaker:')
for state, behavior in states:
    print(f'  {state:10s}: {behavior}')
print()
print('Trip per-TOOL not per-server: one flaky tool does not down the whole MCP.')

</details>

---

## Ex 4: Structured Error Result

**📖 Concept:** When a tool fails, return structured JSON the LLM can reason about (kind, retry hint, suggested action) - never a raw stack trace. The stack trace stays server-side.

**📝 Task:** Build `tool_error` with the listed keys, then print it with `json.dumps(..., indent=2)`.

**📤 Expected:** An indented JSON object with `status=error` and an actionable `error_kind` / retry hint.

In [ ]:
# ✏️ YOUR CODE HERE
import json

# TODO: build `tool_error`, a dict the LLM can reason about (NOT a stack trace),
# with keys: status, error_kind, message, retry_after_seconds, suggested_action, trace_id
# then print it with json.dumps(tool_error, indent=2)
tool_error = {}

<details><summary>💡 Hint</summary>

`json.dumps(tool_error, indent=2)` pretty-prints the dict. Set `status='error'`, an `error_kind` like `'rate_limited'`, and a numeric `retry_after_seconds`.
</details>

**✅ Criteria:** Dict has all six keys, printed as indented JSON, stack trace kept server-side.

<details><summary>✅ Solution</summary>



In [ ]:
print('Tool error result the LLM can reason about (NOT a stack trace):')
import json
tool_error = {
    'status': 'error',
    'error_kind': 'rate_limited',
    'message': 'Per-tool rate limit; retry in 5s.',
    'retry_after_seconds': 5,
    'suggested_action': 'wait + retry, or skip this tool',
    'trace_id': '7f3a-...',
}
print(json.dumps(tool_error, indent=2))
print()
print('Server logs the stack trace separately; LLM sees actionable JSON.')

</details>

---

## Ex 5: Structured JSON Log

**📖 Concept:** Every tool call emits one structured JSON log line, queryable by any field (tenant, tool, latency, trace_id) - never free-text.

**📝 Task:** Build `log` with the listed keys, then print it as compact JSON with `json.dumps(log)`.

**📤 Expected:** A single-line JSON object carrying tenant, client_id, tool, latency_ms, status, and trace context.

In [ ]:
# ✏️ YOUR CODE HERE
import json

# TODO: build `log`, a single structured log line for one tool call, with keys:
#   ts, event, tenant, client_id, tool, latency_ms, status, trace_id, parent_span
# then print it as compact JSON with json.dumps(log)
log = {}

<details><summary>💡 Hint</summary>

Compact means no `indent` argument, so log shippers parse one line per event. Include `trace_id` and `parent_span` so calls can be stitched into a trace.
</details>

**✅ Criteria:** Dict has all nine keys, printed as one-line JSON.

<details><summary>✅ Solution</summary>



In [ ]:
print('Per-call JSON log line:')
import json
log = {
    'ts': '2026-04-18T10:42:15Z',
    'event': 'tool_call',
    'tenant': 'acme',
    'client_id': 'usr_042',
    'tool': 'web_search',
    'latency_ms': 412,
    'status': 'ok',
    'trace_id': '7f3a-...',
    'parent_span': 'b210-...',
}
print(json.dumps(log))
print()
print('Query in BigQuery / Elasticsearch / Loki by any field. Never free-text logs.')

</details>

---

## Ex 6: Health Check Tiers

**📖 Concept:** Health checks come in three tiers - liveness (is the process up?), readiness (are deps reachable?), and tool-level (per-tool success rate). Kubernetes uses the first two to restart or drain pods.

**📝 Task:** Print each tier, its checks, and its k8s use from `tiers`.

**📤 Expected:** Three rows (liveness / readiness / tool-level) with aligned columns.

In [ ]:
# ✏️ YOUR CODE HERE
tiers = [
    ('liveness',  'process responsive (HTTP 200 on /healthz)',     'restarts pod on fail'),
    ('readiness', 'deps reachable (DB ping, downstream API)',      'drains traffic on fail'),
    ('tool-level','per-tool last-success ts + success rate',        'feeds breaker + dashboards'),
]

# TODO: print each tier row as
#   f'  {tier:10s}: {checks:48s} | k8s: {k8s_use}'

<details><summary>💡 Hint</summary>

`for tier, checks, k8s_use in tiers:` and pad with `:10s` / `:48s` so the k8s column lines up.
</details>

**✅ Criteria:** Three tiers printed, k8s use shown for each.

<details><summary>✅ Solution</summary>



In [ ]:
print('Three health-check tiers for MCP:')
for tier, checks, k8s_use in [
    ('liveness',  'process responsive (HTTP 200 on /healthz)',     'restarts pod on fail'),
    ('readiness', 'deps reachable (DB ping, downstream API)',      'drains traffic on fail'),
    ('tool-level','per-tool last-success ts + success rate',        'feeds breaker + dashboards'),
]:
    print(f'  {tier:10s}: {checks:48s} | k8s: {k8s_use}')

</details>

---

## Ex 7: Auth + Multi-Tenancy Checklist

**📖 Concept:** Production auth is layered (OAuth for users, API keys for services, mTLS for zero-trust) and every tool call must be scoped and tenant-isolated. Cross-tenant leakage is a SEV1.

**📝 Task:** Print the `checklist` as a numbered list starting at 1.

**📤 Expected:** Seven numbered items covering auth, scoping, rotation, and tenant isolation.

In [ ]:
# ✏️ YOUR CODE HERE
checklist = [
    'OAuth 2.0 for user-context tools; API key for service-to-service; mTLS for zero-trust',
    'Scoped tokens (per-tool allowlist baked into token claims)',
    'Short-lived tokens (1h-1d), rotation supported',
    'Per-tenant rate limit + per-tenant audit log',
    'Tool implementations MUST filter by tenant_id (never trust)',
    'tenant_id propagated through every span + log',
    'Cross-tenant leakage is a SEV1; nightly isolation tests (12.4)',
]

# TODO: print each item numbered from 1 as f'  {n}. {item}'

<details><summary>💡 Hint</summary>

`enumerate(checklist, 1)` starts the counter at 1: `for n, item in enumerate(checklist, 1):`.
</details>

**✅ Criteria:** All seven items printed and numbered from 1.

<details><summary>✅ Solution</summary>



In [ ]:
print('Production auth + multi-tenant checklist:')
for n, item in enumerate([
    'OAuth 2.0 for user-context tools; API key for service-to-service; mTLS for zero-trust',
    'Scoped tokens (per-tool allowlist baked into token claims)',
    'Short-lived tokens (1h-1d), rotation supported',
    'Per-tenant rate limit + per-tenant audit log',
    'Tool implementations MUST filter by tenant_id (never trust)',
    'tenant_id propagated through every span + log',
    'Cross-tenant leakage is a SEV1; nightly isolation tests (12.4)',
], 1):
    print(f'  {n}. {item}')

</details>

---

---
## Recap
5 pillars: rate limit + error recovery + structured logging + observability + security.
Per-CLIENT token bucket; per-TOOL circuit breaker; per-TENANT isolation.
Structured JSON errors the LLM can reason about; stack traces stay server-side.
Semver tool names with 30-90d deprecation. Health: liveness + readiness + per-tool.
End of Module 10.
